In [63]:
import pandas as pd

In [64]:
player_value_df = pd.read_csv('../data/transfermarkt_data.csv')
player_value_df['Name'] = player_value_df['Name'].replace('Heung-min Son', 'Son Heung-min')
player_value_df.head(3)

,#,Date of birth/Age,Nat.,Height,Foot,Joined,Signed from,Contract,Market value,Name,Position,Team
0,1.0,"Oct 7, 1996 (28)",NaN,"1,94m",right,"Jul 1, 2023",NaN,"Jun 30, 2028",€32.00m,Guglielmo Vicario,Goalkeeper,tottenham-hotspur
1,31.0,"Mar 13, 2003 (22)",NaN,"1,90m",right,"Jan 5, 2025",NaN,"Jun 30, 2031",€15.00m,Antonín Kinský,Goalkeeper,tottenham-hotspur
2,20.0,"Mar 17, 1988 (37)",NaN,"2,01m",right,"Jul 1, 2022",NaN,"Jun 30, 2025",€1.00m,Fraser Forster,Goalkeeper,tottenham-hotspur


In [65]:
player_value_df.columns

Index(['#', 'Date of birth/Age', 'Nat.', 'Height', 'Foot', 'Joined',
       'Signed from', 'Contract', 'Market value', 'Name', 'Position', 'Team'],
      dtype='object')

In [66]:
player_stats_df = pd.read_csv('../data/fbref_data/stats_standard_9.csv')
player_stats_df.head(3)

,Unnamed: 0_level_0_Player,Unnamed: 1_level_0_Nation,Unnamed: 2_level_0_Pos,Unnamed: 3_level_0_Age,Unnamed: 4_level_0_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,...,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Per 90 Minutes_xG,Per 90 Minutes_xAG,Per 90 Minutes_xG+xAG,Per 90 Minutes_npxG,Per 90 Minutes_npxG+xAG,Unnamed: 33_level_0_Matches,Team,Playing Time_MP
0,Pedro Porro,es ESP,DF,24.0,33.0,28,2610.0,29.0,2.0,6.0,...,0.07,0.28,0.06,0.17,0.23,0.06,0.23,Matches,Tottenham,NaN
1,Dejan Kulusevski,se SWE,"MF,FW",24.0,32.0,27,2389.0,26.5,7.0,4.0,...,0.26,0.41,0.16,0.23,0.39,0.16,0.39,Matches,Tottenham,NaN
2,Dominic Solanke,eng ENG,FW,26.0,27.0,25,2201.0,24.5,9.0,3.0,...,0.33,0.45,0.45,0.12,0.57,0.41,0.54,Matches,Tottenham,NaN


In [67]:
player_stats_df.columns

Index(['Unnamed: 0_level_0_Player', 'Unnamed: 1_level_0_Nation',
       'Unnamed: 2_level_0_Pos', 'Unnamed: 3_level_0_Age',
       'Unnamed: 4_level_0_MP', 'Playing Time_Starts', 'Playing Time_Min',
       'Playing Time_90s', 'Performance_Gls', 'Performance_Ast',
       'Performance_G+A', 'Performance_G-PK', 'Performance_PK',
       'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR',
       'Expected_xG', 'Expected_npxG', 'Expected_xAG', 'Expected_npxG+xAG',
       'Progression_PrgC', 'Progression_PrgP', 'Progression_PrgR',
       'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A',
       'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK', 'Per 90 Minutes_xG',
       'Per 90 Minutes_xAG', 'Per 90 Minutes_xG+xAG', 'Per 90 Minutes_npxG',
       'Per 90 Minutes_npxG+xAG', 'Unnamed: 33_level_0_Matches', 'Team',
       'Playing Time_MP'],
      dtype='object')

In [68]:
def parse_market_value(val):
    if isinstance(val, str):
        val = val.replace('€', '').replace('m', '').replace('Th.', '').replace(',', '').strip()
        if 'm' in val:
            return float(val.replace('m', '')) * 1_000_000
        elif 'Th.' in val:
            return float(val.replace('Th.', '')) * 1_000
        else:
            try:
                return float(val)
            except ValueError:
                return None
    return val

In [69]:
# Select relevant columns
value_cols = player_value_df[['Name', 'Market value', 'Team']]
stats_cols = player_stats_df[['Unnamed: 0_level_0_Player', 'Expected_xG', 'Expected_xAG']]

# Merge on player name columns
model_df = pd.merge(
    value_cols,
    stats_cols,
    left_on='Name',
    right_on='Unnamed: 0_level_0_Player',
    how='inner'
)

model_df['Market value'] = model_df['Market value'].apply(parse_market_value)
model_df = model_df.dropna(subset=['Market value'])

# Optional: drop duplicate name columns if desired
model_df = model_df.drop(columns=['Unnamed: 0_level_0_Player'])

model_df.head(3)

,Name,Market value,Team,Expected_xG,Expected_xAG
0,Guglielmo Vicario,32.0,tottenham-hotspur,0.0,0.0
1,Antonín Kinský,15.0,tottenham-hotspur,0.0,0.1
2,Fraser Forster,1.0,tottenham-hotspur,0.0,0.0


In [70]:
model_df.Team.value_counts()

Team
tottenham-hotspur    28
brentford-fc         19
Name: count, dtype: int64

In [71]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Full mapping: Brentford -> Spurs counterparts
mapping = {
    'Mark Flekken': 'Guglielmo Vicario',
    'Keane Lewis-Potter': 'Destiny Udogie',
    'Sepp van den Berg': 'Micky van de Ven',
    'Nathan Collins': 'Cristian Romero',
    'Michael Kayode': 'Pedro Porro',
    'Yehor Yarmoliuk': 'Rodrigo Bentancur',
    'Christian Nørgaard': 'Yves Bissouma',
    'Mikkel Damsgaard': 'James Maddison',
    'Bryan Mbeumo': 'Dejan Kulusevski',
    'Kevin Schade': 'Son Heung-min',
    'Yoane Wissa': 'Dominic Solanke'
}

# Label system for each record in model_df
model_df = model_df.copy()
model_df['system'] = model_df['Name'].apply(
    lambda n: 'Spurs' if n in mapping.values() else ('Brentford' if n in mapping.keys() else None)
)
# Drop any rows outside our mapping
model_df = model_df.dropna(subset=['system'])

# Prepare training DataFrame
df_train = model_df[['Market value', 'system', 'Expected_xG', 'Expected_xAG']].copy()
# Encode system binary: 0 for Spurs, 1 for Brentford
df_train['system_bin'] = df_train['system'].map({'Spurs': 0, 'Brentford': 1})

# Train linear models for xG and xAG
X = df_train[['Market value', 'system_bin']]
model_xg = LinearRegression().fit(X, df_train['Expected_xG'])
model_xag = LinearRegression().fit(X, df_train['Expected_xAG'])

# Prepare prediction set: Spurs players in Brentford system
pred_list = []
for bf_player, spurs_player in mapping.items():
    match = model_df.loc[model_df['Name'] == spurs_player, 'Market value']
    if not match.empty:
        val = match.iloc[0]
        pred_list.append({'Name': spurs_player, 'Market value': val, 'system_bin': 1})
    else:
        print(f"Warning: {spurs_player} not found in model_df['Name']")

df_pred = pd.DataFrame(pred_list)
X_pred = df_pred[['Market value', 'system_bin']]

# Predict estimated stats
df_pred['Estimated_xG_in_Brentford'] = model_xg.predict(X_pred)
df_pred['Estimated_xAG_in_Brentford'] = model_xag.predict(X_pred)

# Merge with actual Spurs Expected stats
df_actual = model_df[model_df['system'] == 'Spurs'][['Name', 'Expected_xG', 'Expected_xAG']]
result = df_actual.merge(
    df_pred[['Name', 'Estimated_xG_in_Brentford', 'Estimated_xAG_in_Brentford']],
    on='Name'
)

# Compute deltas
result['Delta_xG'] = result['Estimated_xG_in_Brentford'] - result['Expected_xG']
result['Delta_xAG'] = result['Estimated_xAG_in_Brentford'] - result['Expected_xAG']

# Reorder and rename for clarity
result = result.rename(columns={
    'Name': 'Spurs Player',
    'Expected_xG': 'Spurs_xG',
    'Expected_xAG': 'Spurs_xAG'
})
result = result[['Spurs Player', 'Spurs_xG', 'Estimated_xG_in_Brentford', 'Delta_xG',
                 'Spurs_xAG', 'Estimated_xAG_in_Brentford', 'Delta_xAG']]

# Save and display
result.to_csv('spurs_brentford_xg_xag_estimates.csv', index=False)
result

,Spurs Player,Spurs_xG,Estimated_xG_in_Brentford,Delta_xG,Spurs_xAG,Estimated_xAG_in_Brentford,Delta_xAG
0,Guglielmo Vicario,0.0,6.444340,6.444340,0.0,3.732249,3.732249
1,Cristian Romero,1.6,9.456738,7.856738,0.6,5.310103,4.710103
2,Micky van de Ven,0.6,9.456738,8.856738,1.2,5.310103,4.110103
3,Destiny Udogie,0.3,7.783184,7.483184,1.5,4.433518,2.933518
4,Pedro Porro,1.6,7.448473,5.848473,5.0,4.258200,-0.741800
5,Rodrigo Bentancur,1.3,6.109629,4.809629,0.4,3.556932,3.156932
6,Yves Bissouma,0.6,5.272851,4.672851,0.3,3.118639,2.818639
7,Dejan Kulusevski,4.2,9.456738,5.256738,6.2,5.310103,-0.889897
8,James Maddison,5.8,8.117894,2.317894,4.2,4.608835,0.408835
9,Son Heung-min,7.2,4.436074,-2.763926,8.2,2.680346,-5.519654


In [73]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Build the regression dataset
reg_rows = []
for bf_player, spurs_player in mapping.items():
    bf_row = model_df[model_df['Name'] == bf_player]
    spurs_row = model_df[model_df['Name'] == spurs_player]
    if not bf_row.empty and not spurs_row.empty:
        reg_rows.append({
            'bf_xg': bf_row['Expected_xG'].iloc[0],
            'bf_xag': bf_row['Expected_xAG'].iloc[0],
            'bf_mv': bf_row['Market value'].iloc[0],
            'spurs_mv': spurs_row['Market value'].iloc[0],
            'spurs_xg': spurs_row['Expected_xG'].iloc[0],
            'spurs_xag': spurs_row['Expected_xAG'].iloc[0],
            'Brentford Player': bf_player,
            'Spurs Player': spurs_player
        })
    else:
        print(f"Missing data for: {bf_player} or {spurs_player}")

reg_df = pd.DataFrame(reg_rows)

# Fit regression for xG
X = reg_df[['bf_mv', 'spurs_mv']]
y_xg = reg_df['bf_xg']
model_xg = LinearRegression().fit(X, y_xg)

# Predict what each Spurs player would get in Brentford's system
reg_df['Estimated_xG_in_Brentford'] = model_xg.predict(reg_df[['bf_mv', 'spurs_mv']])

# Fit regression for xAG
y_xag = reg_df['bf_xag']
model_xag = LinearRegression().fit(X, y_xag)
reg_df['Estimated_xAG_in_Brentford'] = model_xag.predict(reg_df[['bf_mv', 'spurs_mv']])

# Calculate deltas
reg_df['Delta_xG'] = reg_df['Estimated_xG_in_Brentford'] - reg_df['spurs_xg']
reg_df['Delta_xAG'] = reg_df['Estimated_xAG_in_Brentford'] - reg_df['spurs_xag']

# Select and rename columns for output
result = reg_df[['Spurs Player', 'Brentford Player',
                 'spurs_xg', 'bf_xg', 'Estimated_xG_in_Brentford', 'Delta_xG',
                 'spurs_xag', 'bf_xag', 'Estimated_xAG_in_Brentford', 'Delta_xAG',
                 'spurs_mv', 'bf_mv']]

result = result.rename(columns={
    'spurs_xg': 'Spurs_xG',
    'bf_xg': 'Brentford_xG',
    'spurs_xag': 'Spurs_xAG',
    'bf_xag': 'Brentford_xAG',
    'spurs_mv': 'Spurs_MarketValue',
    'bf_mv': 'Brentford_MarketValue'
})

result.to_csv('spurs_brentford_xg_xag_estimates.csv', index=False)
result

Missing data for: Yehor Yarmoliuk or Rodrigo Bentancur


,Spurs Player,Brentford Player,Spurs_xG,Brentford_xG,Estimated_xG_in_Brentford,Delta_xG,Spurs_xAG,Brentford_xAG,Estimated_xAG_in_Brentford,Delta_xAG,Spurs_MarketValue,Brentford_MarketValue
0,Guglielmo Vicario,Mark Flekken,0.0,0.0,0.796003,0.796003,0.0,0.5,0.524796,0.524796,32.0,10.0
1,Destiny Udogie,Keane Lewis-Potter,0.3,3.1,4.107756,3.807756,1.5,3.2,2.633037,1.133037,40.0,23.0
2,Micky van de Ven,Sepp van den Berg,0.6,2.0,1.722366,1.122366,1.2,0.8,2.079812,0.879812,50.0,22.0
3,Cristian Romero,Nathan Collins,1.6,2.5,3.991960,2.391960,0.6,1.8,3.188864,2.588864,50.0,28.0
4,Pedro Porro,Michael Kayode,1.6,0.2,2.617853,1.017853,5.0,1.2,1.782503,-3.217497,38.0,18.0
5,Yves Bissouma,Christian Nørgaard,0.6,4.3,2.579256,1.979256,0.3,1.3,0.967506,0.667506,25.0,11.0
6,James Maddison,Mikkel Damsgaard,5.8,2.8,5.597660,-0.202340,4.2,8.4,3.483570,-0.716430,42.0,28.0
7,Dejan Kulusevski,Bryan Mbeumo,4.2,12.3,14.205133,10.005133,6.2,8.3,8.179599,1.979599,50.0,55.0
8,Son Heung-min,Kevin Schade,7.2,8.2,10.769866,3.569866,8.2,3.7,4.663696,-3.536304,20.0,30.0
9,Dominic Solanke,Yoane Wissa,10.9,18.5,7.512147,-3.387853,2.9,2.6,4.296615,1.396615,40.0,32.0
